# 02. NLR signal validation and analysis eligibility

이 노트북은 workload metadata와 전력 profile의 1:1 연결, 실패 run 제외, 전체 시계열 품질검사, measured/interpolated/simulated 구분, 분석 시간해상도 정책을 확인합니다. 특성값 자체보다 **어떤 profile을 어떤 조건으로 사용할 수 있는지**를 확정하는 것이 목적입니다.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from nlr_data.readers import read_timeseries
from nlr_data.signal_validation import apply_analysis_policy

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. 전체 파이프라인 결과 불러오기

결과가 없으면 먼저 `python -m nlr_data.feature_extraction --data-root D:\\DC_modeling\\NLR_dataset`를 실행합니다.

In [ ]:
DATASET_ROOT = PROJECT_ROOT.parent / 'NLR_dataset'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'signal_validation'

required = {
    'manifest': OUTPUT_DIR / 'run_manifest.parquet',
    'quality': OUTPUT_DIR / 'profile_quality_report.parquet',
    'features': OUTPUT_DIR / 'workload_features.parquet',
}
missing = [str(path) for path in required.values() if not path.is_file()]
assert not missing, f'파이프라인 결과가 없습니다: {missing}'

manifest = pd.read_parquet(required['manifest'])
quality = pd.read_parquet(required['quality'])
features = pd.read_parquet(required['features'])

print(f'Manifest profiles : {len(manifest):,}')
print(f'Quality records   : {len(quality):,}')
print(f'Feature profiles  : {len(features):,}')

## 2. Workload별 분석 대상과 제외 profile

In [ ]:
eligibility = (
    manifest.groupby(['workload_type', 'eligible'], observed=True)
    .size()
    .rename('profile_count')
    .reset_index()
)
eligibility

In [ ]:
manifest.loc[
    ~manifest['eligible'],
    ['profile_id', 'workload_type', 'request_rate', 'run_completed', 'exclusion_reason', 'relative_path'],
]

## 3. 데이터 생성 근거와 분석 시간해상도

Online-rate의 1 ms 값은 실측 독립표본이 아니라 보간된 profile이므로, feature 계산 전에 0.1초 mean-bin으로 축약합니다.

In [ ]:
resolution_policy = (
    manifest.groupby(
        ['workload_type', 'measurement_origin', 'native_interval_s', 'analysis_interval_s', 'resampling_method'],
        dropna=False,
        observed=True,
    )
    .size()
    .rename('profile_count')
    .reset_index()
)
resolution_policy

## 4. 전체 profile 처리 및 품질 상태

In [ ]:
processing_status = (
    quality.groupby(['workload_type', 'processing_status'], observed=True)
    .size()
    .rename('profile_count')
    .reset_index()
)
processing_status

In [ ]:
quality_reasons = quality.loc[
    quality['processing_status'].ne('ok'),
    ['profile_id', 'processing_status', 'processing_error', 'raw_quality_status', 'raw_quality_reasons', 'analysis_quality_status', 'analysis_quality_reasons'],
]
quality_reasons.head(30)

## 5. 원 해상도와 분석 해상도 비교

In [ ]:
interval_summary = (
    quality.loc[quality['processing_status'].eq('ok')]
    .groupby('workload_type', observed=True)
    .agg(
        profiles=('profile_id', 'size'),
        raw_interval_median_s=('raw_median_interval_s', 'median'),
        analysis_interval_median_s=('analysis_median_interval_s', 'median'),
        raw_rows_median=('raw_rows', 'median'),
        analysis_rows_median=('analysis_rows', 'median'),
        profiles_with_raw_gaps=('raw_large_gap_count', lambda values: (values > 0).sum()),
        profiles_with_analysis_gaps=('analysis_large_gap_count', lambda values: (values > 0).sum()),
    )
    .reset_index()
)
interval_summary

## 6. Workload별 대표 분석 파형

화면 확인을 위해 각 workload의 앞부분만 읽습니다. 전체 feature는 이미 전체 길이로 계산되었습니다.

In [ ]:
eligible_manifest = manifest.loc[manifest['eligible']].copy()
representatives = eligible_manifest.groupby('workload_type', observed=True).first().reset_index()

fig, axes = plt.subplots(len(representatives), 1, figsize=(13, 3 * len(representatives)), squeeze=False)
for axis, (_, record) in zip(axes[:, 0], representatives.iterrows()):
    max_rows = 20_000 if record['workload_type'] == 'online_inference_rate' else 10_000
    raw = read_timeseries(
        DATASET_ROOT / record['relative_path'],
        dataset_root=DATASET_ROOT,
        max_rows=max_rows,
        include_metadata_columns=False,
        validate=False,
    )
    analysis = apply_analysis_policy(raw, record)
    axis.plot(analysis['time_s'], analysis['power_w'] / 1_000, linewidth=1)
    axis.set_title(record['workload_type'])
    axis.set_ylabel('Power [kW]')
axis.set_xlabel('Elapsed time [s]')
plt.tight_layout()

## 7. NLR metadata 전력통계와 재계산 결과 비교

Online-rate는 0.1초 mean-bin 축약 때문에 peak 오차가 다른 workload보다 커질 수 있습니다.

In [ ]:
metadata_consistency = (
    features.groupby('workload_type', observed=True)
    .agg(
        profiles=('profile_id', 'size'),
        median_mean_relative_error=('mean_power_metadata_relative_error', 'median'),
        max_mean_relative_error=('mean_power_metadata_relative_error', 'max'),
        median_peak_relative_error=('peak_power_metadata_relative_error', 'median'),
        max_peak_relative_error=('peak_power_metadata_relative_error', 'max'),
    )
    .reset_index()
)
metadata_consistency

## 8. 분석 사용 원칙

- `run_completed=False` profile은 기본 학습·특성분석에서 제외합니다.
- Aggregated measured profile은 job-level IT power이며 PCC 전력이 아닙니다.
- Online-rate는 1 ms 보간점을 0.1초로 축약한 뒤 사용합니다.
- Whole-facility profile은 DIPLOEE 모의자료이며 60초 이상 시간대 분석에 사용합니다.
- Measured와 simulated profile을 하나의 동일 target으로 섞지 않습니다.
- 본 결과가 확정되면 `workload_features.parquet`을 gray-box parameter 및 uncertainty 후보 선정에 사용합니다.